In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    precision_recall_curve
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = '../../../output/alternative_model/'

print('Libraries loaded ✓')

Libraries loaded ✓


In [2]:
X_train = pd.read_csv(OUTPUT_DIR + 'X_train.csv')
X_val   = pd.read_csv(OUTPUT_DIR + 'X_val.csv')
X_test  = pd.read_csv(OUTPUT_DIR + 'X_test.csv')
y_train = pd.read_csv(OUTPUT_DIR + 'y_train.csv').squeeze()
y_val   = pd.read_csv(OUTPUT_DIR + 'y_val.csv').squeeze()
y_test  = pd.read_csv(OUTPUT_DIR + 'y_test.csv').squeeze()

def add_phase1_features(df):
    df = df.copy()
    df['financial_stress_index'] = df['debt_ratio'] * df['behavior_volatility']
    df['academic_resilience']    = df['gpa_latest'] * df['support_numeric']
    df['risk_compounding']       = (df['severe_behavior_flag']
                                    + df['thin_support_flag']
                                    + df['high_pressure_flag'])
    df['loan_to_maturity_ratio'] = df['loan_amount'] / (df['maturity_score'] + 0.1)
    return df

X_train = add_phase1_features(X_train)
X_val   = add_phase1_features(X_val)
X_test  = add_phase1_features(X_test)

print(f'Data loaded — Train:{X_train.shape}, Val:{X_val.shape}, Test:{X_test.shape}')

Data loaded — Train:(7000, 25), Val:(1500, 25), Test:(1500, 25)


In [3]:
try:
    import shap
    print('shap installed ✓')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap
    print('shap installed ✓')

shap installed ✓


In [4]:
# Load Phase 1 best model (SMOTE + Opt Threshold)
with open(OUTPUT_DIR + 'best_model_phase1.pkl', 'rb') as f:
    model_p1 = pickle.load(f)

# SHAP analysis on validation set
explainer = shap.TreeExplainer(model_p1)
shap_values = explainer.shap_values(X_val)

# Mean absolute SHAP value per feature
shap_importance = pd.DataFrame({
    'feature': X_val.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print('Top 15 features by SHAP importance:')
print(shap_importance.head(15).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
top15 = shap_importance.head(15)
ax.barh(top15['feature'][::-1], top15['mean_abs_shap'][::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Top 15 Features — SHAP Importance')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'phase2_shap_importance.png', dpi=120, bbox_inches='tight')
plt.close()
print('SHAP chart saved ✓')

Top 15 features by SHAP importance:
               feature  mean_abs_shap
   behavior_volatility         0.6040
   behavior_risk_score         0.2008
major_income_potential         0.1458
       support_numeric         0.1332
financial_stress_index         0.1095
            has_buffer         0.0997
            gpa_latest         0.0829
loan_to_maturity_ratio         0.0746
         debt_x_living         0.0702
   academic_resilience         0.0547
           loan_amount         0.0533
         living_status         0.0413
       debt_x_behavior         0.0356
            debt_ratio         0.0299
        debt_x_support         0.0286
SHAP chart saved ✓


In [5]:
def add_phase2_features(df):
    """Add Phase 2 advanced interaction features."""
    df = df.copy()

    # ── Risk concentration score ─────────────────────────────────────────
    # High debt + high behavior risk = exponential risk
    df['risk_concentration'] = df['debt_ratio'] * df['behavior_risk_score']

    # ── GPA adjusted income potential ───────────────────────────────────
    # Good GPA in high-income major = lower risk
    df['gpa_income_adjusted'] = df['gpa_latest'] * df['major_income_potential']

    # ── Debt capacity index ──────────────────────────────────────────────
    # How much debt relative to maturity (ability to repay)
    df['debt_capacity'] = df['loan_amount'] / (df['maturity_score'] * df['gpa_latest'] + 0.1)

    # ── Support-adjusted risk ────────────────────────────────────────────
    # More support = less vulnerable to shock
    df['support_adjusted_shock'] = df['shock_vulnerability'] / (df['support_numeric'] + 0.1)

    # ── Academic-behavior interaction ────────────────────────────────────
    # Good academic but high behavior risk = concerning
    df['academic_behavior_gap'] = df['gpa_latest'] - (df['behavior_risk_score'] / 10)

    # ── Pressure resilience ──────────────────────────────────────────────
    df['pressure_resilience'] = df['maturity_score'] / (df['behavior_under_pressure'] + 0.1)

    return df

X_train_p2 = add_phase2_features(X_train)
X_val_p2   = add_phase2_features(X_val)
X_test_p2  = add_phase2_features(X_test)

new_features = ['risk_concentration', 'gpa_income_adjusted', 'debt_capacity',
                'support_adjusted_shock', 'academic_behavior_gap', 'pressure_resilience']
print(f'Added features: {new_features}')
print(f'Total features: {X_train_p2.shape[1]}')

Added features: ['risk_concentration', 'gpa_income_adjusted', 'debt_capacity', 'support_adjusted_shock', 'academic_behavior_gap', 'pressure_resilience']
Total features: 31


In [6]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print('optuna installed ✓')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print('optuna installed ✓')

optuna installed ✓


In [7]:
# Apply SMOTE on phase 2 features
sm = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = sm.fit_resample(X_train_p2, y_train)
print(f'SMOTE applied → Train size: {len(X_train_sm)}')

# Optuna objective function
def objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 2, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800, step=100),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 20),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':            trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 3.0),
        'objective':        'binary:logistic',
        'eval_metric':      'auc',
        'random_state':     42,
        'early_stopping_rounds': 40,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_sm, y_train_sm,
              eval_set=[(X_val_p2, y_val)],
              verbose=False)
    proba = model.predict_proba(X_val_p2)[:, 1]
    return roc_auc_score(y_val, proba)

print('Starting Optuna (50 trials, ~2-3 min)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=False)

print(f'\nBest AUC (val): {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

SMOTE applied → Train size: 8990
Starting Optuna (50 trials, ~2-3 min)...

Best AUC (val): 0.7108
Best params:
  max_depth: 3
  learning_rate: 0.10425900098818623
  n_estimators: 200
  min_child_weight: 20
  subsample: 0.6051201001159638
  colsample_bytree: 0.7708481632492534
  gamma: 0.06779913351859826
  reg_alpha: 1.2349421998680534
  reg_lambda: 0.7382318988613381


In [8]:
best_params = study.best_params.copy()
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42,
    'early_stopping_rounds': 50,
})

xgb_final = xgb.XGBClassifier(**best_params)
xgb_final.fit(X_train_sm, y_train_sm,
              eval_set=[(X_val_p2, y_val)],
              verbose=100)

print('Final model trained ✓')

[0]	validation_0-auc:0.67151
[73]	validation_0-auc:0.70867
Final model trained ✓


In [9]:
# Find optimal threshold
def evaluate_model(name, model, X, y, threshold=0.5):
    proba = model.predict_proba(X)[:, 1]
    preds = (proba >= threshold).astype(int)
    return {
        'Model': name,
        'AUC':       round(roc_auc_score(y, proba), 4),
        'F1':        round(f1_score(y, preds), 4),
        'Precision': round(precision_score(y, preds, zero_division=0), 4),
        'Recall':    round(recall_score(y, preds), 4),
        'Threshold': threshold,
    }

val_proba = xgb_final.predict_proba(X_val_p2)[:, 1]
prec_arr, rec_arr, thr_arr = precision_recall_curve(y_val, val_proba)
f1_arr = 2*(prec_arr[:-1]*rec_arr[:-1]) / (prec_arr[:-1]+rec_arr[:-1]+1e-8)
best_threshold = thr_arr[f1_arr.argmax()]
print(f'Optimal threshold: {best_threshold:.4f}')

Optimal threshold: 0.4425


In [10]:
# Load Phase 1 results for comparison
with open(OUTPUT_DIR + 'best_model_phase1.pkl', 'rb') as f:
    model_p1 = pickle.load(f)
with open(OUTPUT_DIR + 'best_threshold_phase1.pkl', 'rb') as f:
    threshold_p1 = pickle.load(f)

# Phase 1 needs Phase1 features (25 cols)
# Phase 2 model needs Phase2 features (31 cols)

print('=' * 65)
print('FINAL COMPARISON ON TEST SET')
print('=' * 65)

results = [
    evaluate_model('Phase1 Best (SMOTE+Thr)',  model_p1,    X_test,    y_test, threshold_p1),
    evaluate_model('Phase2 (0.5)',             xgb_final,   X_test_p2, y_test, 0.5),
    evaluate_model(f'Phase2 + Opt.Thr ({best_threshold:.3f})', xgb_final, X_test_p2, y_test, best_threshold),
]

df = pd.DataFrame(results).set_index('Model')[['AUC','Precision','Recall','F1']]
print(df.to_string())

print('\nTargets: AUC≥0.75 | Precision≥0.55 | F1≥0.62')

FINAL COMPARISON ON TEST SET
                            AUC  Precision  Recall     F1
Model                                                    
Phase1 Best (SMOTE+Thr)  0.7182     0.4826  0.8004 0.6021
Phase2 (0.5)             0.7158     0.5473  0.5504 0.5488
Phase2 + Opt.Thr (0.442) 0.7158     0.5000  0.6959 0.5819

Targets: AUC≥0.75 | Precision≥0.55 | F1≥0.62


In [11]:
# Comparison chart
metrics_list = ['AUC', 'Precision', 'Recall', 'F1']
targets_dict = {'AUC': 0.75, 'Precision': 0.55, 'Recall': 0.65, 'F1': 0.62}
colors = ['#fd7e14', '#198754', '#0d6efd']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, metric in enumerate(metrics_list):
    ax = axes[i]
    vals = [r[metric] for r in results]
    bars = ax.bar(range(len(results)), vals, color=colors)
    ax.axhline(targets_dict[metric], color='red', ls='--', lw=1.5, label=f'Target: {targets_dict[metric]}')
    ax.set_title(metric, fontweight='bold')
    ax.set_xticks(range(len(results)))
    ax.set_xticklabels(['P1','P2','P2+Thr'], fontsize=8)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', fontsize=7)

plt.suptitle('Phase 2 — Model Comparison (Test Set)', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'phase2_comparison.png', dpi=120, bbox_inches='tight')
plt.close()
print('Chart saved ✓')

Chart saved ✓


In [12]:
best = max(results, key=lambda x: x['F1'])
print(f'Best Phase2 model: {best["Model"]}')
print(f'  AUC={best["AUC"]:.4f}  Precision={best["Precision"]:.4f}  '
      f'Recall={best["Recall"]:.4f}  F1={best["F1"]:.4f}')

# Save
with open(OUTPUT_DIR + 'best_model_phase2.pkl', 'wb') as f:
    pickle.dump(xgb_final, f)
with open(OUTPUT_DIR + 'best_threshold_phase2.pkl', 'wb') as f:
    pickle.dump(best['Threshold'], f)

# Save the feature engineering function info
phase2_meta = {
    'phase1_features': ['financial_stress_index', 'academic_resilience',
                        'risk_compounding', 'loan_to_maturity_ratio'],
    'phase2_features': ['risk_concentration', 'gpa_income_adjusted', 'debt_capacity',
                        'support_adjusted_shock', 'academic_behavior_gap', 'pressure_resilience'],
    'best_threshold': best['Threshold'],
    'best_params': best_params,
}
import json
with open(OUTPUT_DIR + 'phase2_meta.json', 'w') as f:
    json.dump({k: (float(v) if isinstance(v, (np.float64, np.float32)) else v)
               for k, v in phase2_meta.items()}, f, indent=2)

print(f'\nSaved: best_model_phase2.pkl')
print(f'Saved: best_threshold_phase2.pkl (value={best["Threshold"]:.4f})')
print(f'Saved: phase2_meta.json')

# Final summary
baseline = evaluate_model('Baseline', model_p1, X_test, y_test, 0.5)
print('\n' + '='*65)
print('PHASE 2 SUMMARY (vs Baseline)')
print('='*65)
print(f'  Baseline  → AUC:{baseline["AUC"]}  Prec:{baseline["Precision"]}  F1:{baseline["F1"]}')
print(f'  Phase 2   → AUC:{best["AUC"]}  Prec:{best["Precision"]}  F1:{best["F1"]}')
print(f'  ΔAUC:{best["AUC"]-baseline["AUC"]:+.4f}  ΔPrec:{best["Precision"]-baseline["Precision"]:+.4f}  ΔF1:{best["F1"]-baseline["F1"]:+.4f}')

Best Phase2 model: Phase1 Best (SMOTE+Thr)
  AUC=0.7182  Precision=0.4826  Recall=0.8004  F1=0.6021

Saved: best_model_phase2.pkl
Saved: best_threshold_phase2.pkl (value=0.3623)
Saved: phase2_meta.json

PHASE 2 SUMMARY (vs Baseline)
  Baseline  → AUC:0.7182  Prec:0.5626  F1:0.5448
  Phase 2   → AUC:0.7182  Prec:0.4826  F1:0.6021
  ΔAUC:+0.0000  ΔPrec:-0.0800  ΔF1:+0.0573
